# 03 — Relationships, Cardinality & Join Coverage

**Objetivo:** validar empíricamente las relaciones entre las siete fuentes antes de materializar joins.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Métricas
- **Coverage:** proporción de filas del lado izquierdo con match.
- **Multiplicity:** cuántas filas del lado derecho puede tener una key.
- **Expansion factor:** filas esperadas tras LEFT JOIN / filas originales.

Un join que ejecuta sin error no es necesariamente un join correcto.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import (
    discover_dataset_files,
    load_registry,
    load_relationship_candidates,
)

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
RELATIONSHIP_CANDIDATES = load_relationship_candidates(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}


In [ ]:
from argos.common.reports import write_report
from argos.profiling.relationships import (
    analyze_all_relationships,
    analyze_relationship_candidates,
    select_relationship_candidate,
)

cfg = PROJECT_CONFIG["profiling"]
gate_cfg = PROJECT_CONFIG["m1_gate"]

# Resolve contracts -> processes empirically instead of hard-coding the failed
# Proceso de Compra -> ID del Proceso hypothesis.
candidate_spec = RELATIONSHIP_CANDIDATES["contracts_processes"]
relationship_candidates = analyze_relationship_candidates(
    key_prefix=candidate_spec.key,
    left_dataset=candidate_spec.left_dataset,
    right_dataset=candidate_spec.right_dataset,
    left_keys=candidate_spec.left_keys,
    right_key_candidates=candidate_spec.right_key_candidates,
    files=FILES,
    sample_size=cfg["csv_sample_size"],
    min_coverage=gate_cfg["relationship_candidate_min_coverage"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

selected_contracts_processes, relationship_resolution = select_relationship_candidate(
    relationship_candidates,
    relationship_key="contracts_processes",
    min_coverage=gate_cfg["relationship_candidate_min_coverage"],
)

write_report(
    relationship_candidates,
    paths.reports / "profiling" / "relationship_candidate_analysis.parquet",
)
write_report(
    relationship_resolution,
    paths.reports / "profiling" / "relationship_resolution.parquet",
)

relationship_candidates[[
    "candidate_right_keys",
    "status",
    "left_row_coverage",
    "max_right_multiplicity",
    "expansion_factor",
    "requires_right_canonicalization",
]]


In [ ]:
from argos.profiling.grain import analyze_duplicate_structure

# Characterize the exact process key selected by the relationship resolver.
# This is the grain evidence Silver will use before canonicalizing Processes.
selected_process_key = tuple(selected_contracts_processes.right_keys)
selected_process_grain = pd.DataFrame([
    analyze_duplicate_structure(
        "processes",
        FILES["processes"],
        selected_process_key,
        sample_size=cfg["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
])

grain_path = paths.reports / "profiling" / "grain_resolution.parquet"
existing_grain = pd.read_parquet(grain_path) if grain_path.exists() else pd.DataFrame()
selected_key_label = "+".join(selected_process_key)
if not existing_grain.empty:
    existing_grain = existing_grain.loc[
        ~((existing_grain["dataset"] == "processes") & (existing_grain["key"] == selected_key_label))
    ]
grain_resolution = pd.concat([existing_grain, selected_process_grain], ignore_index=True)
write_report(grain_resolution, grain_path)

# Re-run the relationship audit with the empirically selected contracts->processes key.
resolved_relationships = dict(RELATIONSHIPS)
resolved_relationships["contracts_processes"] = selected_contracts_processes

join_analysis = analyze_all_relationships(
    resolved_relationships,
    FILES,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    join_analysis,
    paths.reports / "profiling" / "join_analysis.parquet",
)

display(relationship_resolution)
display(selected_process_grain)
join_analysis[[
    "relationship", "status", "left_dataset", "right_dataset",
    "left_keys", "right_keys", "left_row_coverage",
    "max_right_multiplicity", "expansion_factor",
]]


## Criterio de salida
`docs/data-model.md` debe distinguir **relaciones verificadas** de **hipótesis todavía no aceptadas**.
No construyas una mega-tabla en Bronze.

In [ ]:
# Review, do not automatically reject, relations with high raw multiplicity.
# Processes is known to be non-unique in Bronze; Silver must canonicalize it
# before materializing one-to-one/many-to-one joins.
join_analysis.loc[
    (join_analysis["status"] != "analyzed")
    | (join_analysis["left_row_coverage"].fillna(0) < 0.90)
    | (join_analysis["expansion_factor"].fillna(1) > 1.05)
]
